In [1]:
# CELL 1: Imports and load raw data
# ============================================
import pandas as pd
import numpy as np

df_chargers = pd.read_csv("../data/ev_charging_stations.csv")

print("Chargers shape:", df_chargers.shape)
df_chargers.head()

Chargers shape: (1966, 15)


,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
0,509422,Narketpally - Addanki - Medarametla Road,16.890620,79.561206,MIRYALGUDA,Telangana,508207,(Unknown Operator),Operational,True,180.0,6,True,2026-09-16T10:37:00Z,2026-09-19T10:39:00Z
1,508929,Rajahmundry unit -1,17.039631,81.615838,Rajahmundry,NaN,534313,(Business Owner at Location),Operational,True,240.0,7,True,2026-09-11T08:38:00Z,2026-09-20T04:18:00Z
2,508928,Kakinada Unit 1,16.982049,82.246026,Kakinada,NaN,533003,(Business Owner at Location),Operational,True,60.0,2,True,2026-09-11T08:38:00Z,2026-09-20T04:18:00Z
3,508927,May Fair Green Mark Project,17.484253,78.212304,NaN,NaN,NaN,Salzer NexCharge (IN),Operational,True,7.0,3,False,2026-09-11T08:37:00Z,2026-09-20T04:18:00Z
4,508926,May Fair Marketing Office,17.483787,78.215093,NaN,NaN,NaN,Salzer NexCharge (IN),Operational,True,7.0,1,False,2026-09-11T08:37:00Z,2026-09-20T04:18:00Z


In [2]:
# CELL 2: Initial inspection — understand what you're working with
# ============================================
print("Column data types:")
print(df_chargers.dtypes)

print("\nMissing values per column:")
print(df_chargers.isnull().sum())

print("\nTotal duplicate rows (exact full-row duplicates):")
print(df_chargers.duplicated().sum())

Column data types:
ID                      int64
name                      str
latitude              float64
longitude             float64
town                      str
state                     str
postcode                  str
operator                  str
status                    str
is_operational           bool
max_power_kw          float64
total_points            int64
has_fast_charging        bool
date_created              str
date_last_verified        str
dtype: object

Missing values per column:
ID                      0
name                    0
latitude                0
longitude               0
town                  434
state                  29
postcode              466
operator              390
status                  0
is_operational          0
max_power_kw          108
total_points            0
has_fast_charging       0
date_created            0
date_last_verified      0
dtype: int64

Total duplicate rows (exact full-row duplicates):
0


In [3]:
df_chargers.info()

<class 'pandas.DataFrame'>
RangeIndex: 1966 entries, 0 to 1965
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  1966 non-null   int64  
 1   name                1966 non-null   str    
 2   latitude            1966 non-null   float64
 3   longitude           1966 non-null   float64
 4   town                1532 non-null   str    
 5   state               1937 non-null   str    
 6   postcode            1500 non-null   str    
 7   operator            1576 non-null   str    
 8   status              1966 non-null   str    
 9   is_operational      1966 non-null   bool   
 10  max_power_kw        1858 non-null   float64
 11  total_points        1966 non-null   int64  
 12  has_fast_charging   1966 non-null   bool   
 13  date_created        1966 non-null   str    
 14  date_last_verified  1966 non-null   str    
dtypes: bool(2), float64(3), int64(2), str(8)
memory usage: 408.9 KB


In [4]:
# ============================================
# CELL 3: Inspect the 'state' column for inconsistencies
# ============================================
print("Unique state values BEFORE cleaning:")
print(sorted(df_chargers["state"].dropna().unique()))

Unique state values BEFORE cleaning:
[' Tamil Nadu', 'Ahmedabad', 'Andhra Pradesh', 'Assam', 'Bangalore Urban', 'Bihar', 'Burari', 'Dadra and Nagar Haveli', 'Delhi', 'Delhi ', 'GJ', 'Goa', 'Gujarat', 'Gujarat ', 'Haryana', 'Haryana ', 'Himachal Pradesh', 'India', 'Jammu and Kashmir', 'Jharkhand', 'KA', 'KARNATAKA', 'Karnatak', 'Karnataka', 'Karnataka ', 'Keraka', 'Keral', 'Kerala', 'Kerala ', 'Lerala', 'MH', 'MP', 'Madhya Pradesh', 'Maharashtra', 'Maharashtra ', 'Maharshtra', 'Mahrashtra', 'New Dehi', 'Odisha', 'Puducherry', 'Punjab ', 'RJ', 'Rajasthan', 'Rajasthan ', 'TAMILNADU', 'Tamil Nadu', 'Tamil Nadu ', 'Tamil Nasdu', 'Tamil nadu', 'TamilNadu', 'Tamilnadu', 'Tamilnadu ', 'Telangana', 'Telangana ', 'UTTAR PRADESH', 'UTTAR PRADESH ', 'Uttar Pradesh', 'Uttar Pradesh ', 'Uttarakhand', 'Uttarakhnad', 'Villupuram', 'West Bengal', 'chennai', 'karnataka', 'rajasthan', 'uttar pradesh ', 'west bengal']


In [5]:
# ============================================
# CELL 6 (UPDATED): Standardize the 'state' column — comprehensive fix
# ============================================

# Step 1: Basic cleanup — strip whitespace, title case
df_chargers["state"] = df_chargers["state"].astype(str).str.strip()

In [6]:
# Step 2: Comprehensive mapping — covers typos, abbreviations, casing, and city-to-state fixes
state_name_map = {
    # Casing/whitespace variants (after strip, these differ only in case)
    "KARNATAKA": "Karnataka",
    "karnataka": "Karnataka",
    "Karnatak": "Karnataka",
    "TAMILNADU": "Tamil Nadu",
    "TamilNadu": "Tamil Nadu",
    "Tamilnadu": "Tamil Nadu",
    "Tamil nadu": "Tamil Nadu",
    "UTTAR PRADESH": "Uttar Pradesh",
    "uttar pradesh": "Uttar Pradesh",
    "rajasthan": "Rajasthan",
    "west bengal": "West Bengal",

    # Abbreviations
    "GJ": "Gujarat",
    "KA": "Karnataka",
    "MH": "Maharashtra",
    "MP": "Madhya Pradesh",
    "RJ": "Rajasthan",

    # Spelling typos
    "Maharshtra": "Maharashtra",
    "Mahrashtra": "Maharashtra",
    "Keraka": "Kerala",
    "Keral": "Kerala",
    "Lerala": "Kerala",
    "Tamil Nasdu": "Tamil Nadu",
    "Uttarakhnad": "Uttarakhand",
    "New Dehi": "Delhi",

    # Cities/districts mistakenly entered as state — mapped to their correct state
    "Ahmedabad": "Gujarat",
    "Bangalore Urban": "Karnataka",
    "Burari": "Delhi",
    "Villupuram": "Tamil Nadu",
    "chennai": "Tamil Nadu",

    # Invalid entry — will be dropped in the next step
    "India": None,
}
df_chargers["state"] = df_chargers["state"].replace(state_name_map)
df_chargers

,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
0,509422,Narketpally - Addanki - Medarametla Road,16.890620,79.561206,MIRYALGUDA,Telangana,508207,(Unknown Operator),Operational,True,180.0,6,True,2026-09-16T10:37:00Z,2026-09-19T10:39:00Z
1,508929,Rajahmundry unit -1,17.039631,81.615838,Rajahmundry,NaN,534313,(Business Owner at Location),Operational,True,240.0,7,True,2026-09-11T08:38:00Z,2026-09-20T04:18:00Z
2,508928,Kakinada Unit 1,16.982049,82.246026,Kakinada,NaN,533003,(Business Owner at Location),Operational,True,60.0,2,True,2026-09-11T08:38:00Z,2026-09-20T04:18:00Z
3,508927,May Fair Green Mark Project,17.484253,78.212304,NaN,NaN,NaN,Salzer NexCharge (IN),Operational,True,7.0,3,False,2026-09-11T08:37:00Z,2026-09-20T04:18:00Z
4,508926,May Fair Marketing Office,17.483787,78.215093,NaN,NaN,NaN,Salzer NexCharge (IN),Operational,True,7.0,1,False,2026-09-11T08:37:00Z,2026-09-20T04:18:00Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1961,103604,"International Tech Park, Bangalore",12.985421,77.738513,Bangalore,Karnataka,560066,(Unknown Operator),Operational,True,3.0,2,False,2018-05-04T08:06:00Z,2018-05-11T11:06:00Z
1962,103603,Ather Grid Point - Ascendas Park square mall,12.987219,77.736474,Bangalore,Karnataka,560066,(Unknown Operator),Operational,True,3.0,2,False,2018-05-04T08:01:00Z,2018-05-11T11:06:00Z
1963,96047,Tellus Power,17.413362,78.426597,Hyderabad,Telangana,500034,NaN,Operational,True,7.2,1,False,2017-12-07T10:12:00Z,2017-12-08T08:52:00Z
1964,92772,Tata Power Receiving Station,19.095796,72.926011,Mumbai,NaN,400077,(Business Owner at Location),Operational,True,NaN,3,False,2017-09-10T11:59:00Z,2017-09-10T11:59:00Z


In [7]:
# Step 3: Now apply title case to catch anything the map missed, and re-strip
df_chargers["state"] = df_chargers["state"].astype(str).str.strip().str.title()
df_chargers["state"] = df_chargers["state"].replace({"None": None, "Nan": None})
df_chargers

,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
0,509422,Narketpally - Addanki - Medarametla Road,16.890620,79.561206,MIRYALGUDA,Telangana,508207,(Unknown Operator),Operational,True,180.0,6,True,2026-09-16T10:37:00Z,2026-09-19T10:39:00Z
1,508929,Rajahmundry unit -1,17.039631,81.615838,Rajahmundry,NaN,534313,(Business Owner at Location),Operational,True,240.0,7,True,2026-09-11T08:38:00Z,2026-09-20T04:18:00Z
2,508928,Kakinada Unit 1,16.982049,82.246026,Kakinada,NaN,533003,(Business Owner at Location),Operational,True,60.0,2,True,2026-09-11T08:38:00Z,2026-09-20T04:18:00Z
3,508927,May Fair Green Mark Project,17.484253,78.212304,NaN,NaN,NaN,Salzer NexCharge (IN),Operational,True,7.0,3,False,2026-09-11T08:37:00Z,2026-09-20T04:18:00Z
4,508926,May Fair Marketing Office,17.483787,78.215093,NaN,NaN,NaN,Salzer NexCharge (IN),Operational,True,7.0,1,False,2026-09-11T08:37:00Z,2026-09-20T04:18:00Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1961,103604,"International Tech Park, Bangalore",12.985421,77.738513,Bangalore,Karnataka,560066,(Unknown Operator),Operational,True,3.0,2,False,2018-05-04T08:06:00Z,2018-05-11T11:06:00Z
1962,103603,Ather Grid Point - Ascendas Park square mall,12.987219,77.736474,Bangalore,Karnataka,560066,(Unknown Operator),Operational,True,3.0,2,False,2018-05-04T08:01:00Z,2018-05-11T11:06:00Z
1963,96047,Tellus Power,17.413362,78.426597,Hyderabad,Telangana,500034,NaN,Operational,True,7.2,1,False,2017-12-07T10:12:00Z,2017-12-08T08:52:00Z
1964,92772,Tata Power Receiving Station,19.095796,72.926011,Mumbai,NaN,400077,(Business Owner at Location),Operational,True,NaN,3,False,2017-09-10T11:59:00Z,2017-09-10T11:59:00Z


In [8]:
# Step 4: Drop rows where state is now invalid/missing (like the "India" entries)
before = df_chargers.shape[0]
df_chargers = df_chargers.dropna(subset=["state"])
after = df_chargers.shape[0]
print(f"Dropped {before - after} rows with invalid/missing state. Remaining: {after}")


Dropped 30 rows with invalid/missing state. Remaining: 1936


In [9]:
# Step 5: Verify the cleanup worked
print("\nUnique state values AFTER cleaning:")
print(sorted(df_chargers["state"].unique()))


Unique state values AFTER cleaning:
['Andhra Pradesh', 'Assam', 'Bihar', 'Dadra And Nagar Haveli', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu And Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Tamil Nadu', 'Telangana', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [10]:
# ============================================
# CELL: Handle missing values (using YOUR actual counts)
# ============================================

# town — 434 missing (22%) — don't drop, just label
df_chargers["town"] = df_chargers["town"].fillna("Unknown")

In [11]:
# state — only 29 missing (1.5%) — small enough to just label rather than drop
df_chargers["state"] = df_chargers["state"].fillna("Unknown")

In [12]:
# postcode — 466 missing (24%) — not critical for your analysis (you have lat/long), just label
df_chargers["postcode"] = df_chargers["postcode"].fillna("Unknown")


In [13]:
# operator — 390 missing (20%) — this is normal for OpenChargeMap crowd data, label clearly
df_chargers["operator"] = df_chargers["operator"].fillna("Unknown Operator")

In [14]:
# max_power_kw — 108 missing (5%) — fill with 0 so you don't lose these station locations
df_chargers["max_power_kw"] = df_chargers["max_power_kw"].fillna(0)


In [15]:
# Confirm no missing values remain in critical columns
print("Missing values after fill:")
print(df_chargers.isnull().sum())

Missing values after fill:
ID                    0
name                  0
latitude              0
longitude             0
town                  0
state                 0
postcode              0
operator              0
status                0
is_operational        0
max_power_kw          0
total_points          0
has_fast_charging     0
date_created          0
date_last_verified    0
dtype: int64


In [16]:
# ============================================
# CELL: Sanity check on power values
# ============================================
print("Power range:", df_chargers["max_power_kw"].min(), "to", df_chargers["max_power_kw"].max())

# Cap unrealistic outliers if any exist
before = df_chargers.shape[0]
df_chargers = df_chargers[df_chargers["max_power_kw"] <= 500]
after = df_chargers.shape[0]
print(f"Removed {before - after} rows with unrealistic power values.")

Power range: 0.0 to 1000000.0
Removed 2 rows with unrealistic power values.


In [17]:
# ============================================
# CELL: Remove duplicate stations
# ============================================
before = df_chargers.shape[0]
df_chargers = df_chargers.drop_duplicates(subset=["latitude", "longitude"])
after = df_chargers.shape[0]
print(f"Removed {before - after} duplicate-location stations. Remaining: {after}")

Removed 97 duplicate-location stations. Remaining: 1837


In [18]:
# 2. Coordinate validation
invalid_coords = df_chargers[
    (~df_chargers["latitude"].between(6, 38)) | 
    (~df_chargers["longitude"].between(68, 98))
]
print("Invalid coordinates:", invalid_coords.shape[0])
invalid_coords

Invalid coordinates: 1


,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
1880,301542,Ero G Electric Charging Station,9.357268,11.301994,Erode,Tamil Nadu,638052,Relux Electric (India),Operational,True,44.0,1,True,2024-07-22T07:31:00Z,2024-07-26T06:15:00Z


In [19]:
# ============================================
# Drop the row with clearly invalid coordinates
# (ID 301542 — "Ero G Electric Charging Station" — coordinates place it in Africa, not Erode, TN)
# ============================================
before = df_chargers.shape[0]
df_chargers = df_chargers[df_chargers["ID"] != 301542]
after = df_chargers.shape[0]
print(f"Dropped {before - after} row(s) with invalid coordinates. Remaining: {after}")

Dropped 1 row(s) with invalid coordinates. Remaining: 1836


In [20]:
# 3. total_points validation
invalid_points = df_chargers[
    (df_chargers["total_points"] <= 0) | (df_chargers["total_points"] > 50)
]
print("Invalid total_points:", invalid_points.shape[0])
invalid_points

Invalid total_points: 6


,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
310,479330,Electric Vehicle Charging Station,21.016109,81.702900,Unknown,Odisha,Unknown,Unknown Operator,Operational,True,0.0,600,False,2026-02-28T08:08:00Z,2026-03-01T09:24:00Z
1434,309318,Jio-BP Pulse Varthur Hub,12.939790,77.734470,Bengaluru,Karnataka,560087,JIO BP Pulse (India),Operational,True,60.0,136,True,2025-03-06T17:40:00Z,2025-03-07T16:59:00Z
1435,309317,"Jio-BP Pulse Hub, Marathalli",12.951790,77.702070,Bengaluru,Karnataka,560037,JIO BP Pulse (India),Operational,True,60.0,98,True,2025-03-06T17:35:00Z,2025-03-07T16:59:00Z
1494,309186,Jio BP Charging Hub Chikkagubbi,13.087848,77.660148,Bengaluru,Karnataka,562149,JIO BP Pulse (India),Operational,True,60.0,56,True,2025-03-03T17:03:00Z,2025-03-06T17:04:00Z
1596,308922,Blusmart Charging Station (HSR Hub),12.902120,77.641910,Bengaluru,Karnataka,560068,JIO BP Pulse (India),Operational,True,60.0,58,True,2025-02-17T13:06:00Z,2025-03-04T03:24:00Z
1598,308920,JioBP Pulse Charging Hub - Electronic City Pha...,12.864620,77.674860,Bengaluru,Karnataka,560100,JIO BP Pulse (India),Operational,True,60.0,70,True,2025-02-17T12:54:00Z,2025-03-04T03:24:00Z


In [21]:
# Look at the actual rows with suspiciously high total_points
suspicious = df_chargers[df_chargers["total_points"] > 50].sort_values("total_points", ascending=False)
print(suspicious.shape[0], "rows found")
suspicious[["ID", "name", "town", "state", "total_points", "max_power_kw"]]

6 rows found


,ID,name,town,state,total_points,max_power_kw
310,479330,Electric Vehicle Charging Station,Unknown,Odisha,600,0.0
1434,309318,Jio-BP Pulse Varthur Hub,Bengaluru,Karnataka,136,60.0
1435,309317,"Jio-BP Pulse Hub, Marathalli",Bengaluru,Karnataka,98,60.0
1598,308920,JioBP Pulse Charging Hub - Electronic City Pha...,Bengaluru,Karnataka,70,60.0
1596,308922,Blusmart Charging Station (HSR Hub),Bengaluru,Karnataka,58,60.0
1494,309186,Jio BP Charging Hub Chikkagubbi,Bengaluru,Karnataka,56,60.0


In [22]:
# Step 1: Drop the one genuinely broken record (Odisha, 600 points, 0kW, unknown town)
before = df_chargers.shape[0]
df_chargers = df_chargers[df_chargers["ID"] != 479330]
after = df_chargers.shape[0]
print(f"Dropped {before - after} clearly invalid record(s). Remaining: {after}")

# Step 2: Keep the Bengaluru hub records as-is — they appear to be legitimate large hubs
# No action needed here, just confirming your max is now reasonable
print("New total_points range:", df_chargers["total_points"].min(), "-", df_chargers["total_points"].max())

Dropped 1 clearly invalid record(s). Remaining: 1835
New total_points range: 1 - 136


In [23]:
# 4. Duplicate ID check
duplicate_ids = df_chargers[df_chargers["ID"].duplicated(keep=False)]
print("Duplicate IDs:", duplicate_ids.shape[0])

Duplicate IDs: 0


In [24]:
inconsistent = df_chargers[
    ((df_chargers["status"] == "Operational") & (df_chargers["is_operational"] == False)) |
    ((df_chargers["status"] == "Not Operational") & (df_chargers["is_operational"] == True))
]
print("Inconsistent status/is_operational rows:", inconsistent.shape[0])
inconsistent[["ID", "name", "status", "is_operational"]]

Inconsistent status/is_operational rows: 0


,ID,name,status,is_operational


In [25]:
# ============================================
# CELL 18b: Near-duplicate coordinate check (FIXED)
# ============================================
df_chargers["lat_rounded"] = df_chargers["latitude"].round(4)
df_chargers["lon_rounded"] = df_chargers["longitude"].round(4)

near_duplicates = df_chargers[df_chargers.duplicated(subset=["lat_rounded", "lon_rounded"], keep=False)]
print("Potential near-duplicate stations:", near_duplicates.shape[0])

# Sort FIRST (while lat_rounded/lon_rounded still exist), THEN select display columns
near_duplicates = near_duplicates.sort_values(["lat_rounded", "lon_rounded"])
near_duplicates[["ID", "name", "latitude", "longitude", "town"]]

Potential near-duplicate stations: 2


,ID,name,latitude,longitude,town
1593,308925,Shivalli Restaurant 1,12.63923,77.14021,Channapatna
1684,308760,Shivalli Chowki Restaurant,12.63923,77.14023,Channapatna


In [26]:
# ============================================
# Check both records in full before deciding which to keep
# ============================================
df_chargers[df_chargers["ID"].isin([308925, 308760])]

,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified,lat_rounded,lon_rounded
1593,308925,Shivalli Restaurant 1,12.63923,77.14021,Channapatna,Karnataka,562160,JIO BP Pulse (India),Operational,True,80.0,2,True,2025-02-17T14:18:00Z,2025-03-04T03:24:00Z,12.6392,77.1402
1684,308760,Shivalli Chowki Restaurant,12.63923,77.14023,Channapatna,Karnataka,562160,Zeon Charging,Operational,True,30.0,2,True,2025-02-11T17:11:00Z,2025-03-04T03:29:00Z,12.6392,77.1402


In [27]:
# Both records confirmed as legitimate distinct chargers at the same location — keeping both
df_chargers = df_chargers.drop(columns=["lat_rounded", "lon_rounded"])
print("Near-duplicate check complete — no rows dropped, confirmed as distinct chargers.")

Near-duplicate check complete — no rows dropped, confirmed as distinct chargers.


In [28]:
print(df_chargers["state"].value_counts())

state
Kerala                    595
Karnataka                 378
Odisha                    284
Tamil Nadu                211
Maharashtra                85
Gujarat                    66
Rajasthan                  47
Delhi                      34
Haryana                    28
Uttar Pradesh              28
Telangana                  17
Uttarakhand                14
Madhya Pradesh             13
West Bengal                 9
Andhra Pradesh              7
Goa                         6
Dadra And Nagar Haveli      3
Assam                       2
Jharkhand                   2
Punjab                      2
Jammu And Kashmir           1
Himachal Pradesh            1
Puducherry                  1
Bihar                       1
Name: count, dtype: int64


In [29]:
# ============================================
# CELL: Convert date columns properly
# ============================================
df_chargers["date_created"] = pd.to_datetime(df_chargers["date_created"], errors="coerce")
df_chargers["date_last_verified"] = pd.to_datetime(df_chargers["date_last_verified"], errors="coerce")

print(df_chargers[["date_created", "date_last_verified"]].dtypes)

date_created          datetime64[us, UTC]
date_last_verified    datetime64[us, UTC]
dtype: object


In [30]:
df_chargers

,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
0,509422,Narketpally - Addanki - Medarametla Road,16.890620,79.561206,MIRYALGUDA,Telangana,508207,(Unknown Operator),Operational,True,180.0,6,True,2026-09-16 10:37:00+00:00,2026-09-19 10:39:00+00:00
17,504729,Uplug Ev Charging Station & Network,16.945913,74.405395,Ashta,Maharashtra,416301,(Unknown Operator),Operational,True,60.0,2,True,2026-08-15 06:26:00+00:00,2026-08-18 06:28:00+00:00
18,503818,NH-7 Hotel,20.634579,78.927604,Unknown,Maharashtra,Unknown,Charge_iN by Mahindra,Operational,True,240.0,4,True,2026-08-09 13:23:00+00:00,2026-08-09 13:23:00+00:00
19,503599,The Big Bay Highway Pitstop,13.605916,77.786870,Unknown,Karnataka,562104,Charge_iN by Mahindra,Operational,True,240.0,4,True,2026-08-08 09:33:00+00:00,2026-08-08 09:33:00+00:00
20,502497,TransEV Charging Station,23.547824,87.272638,Durgapur,West Bengal,713203,(Unknown Operator),Operational,True,60.0,4,True,2026-08-06 06:28:00+00:00,2026-08-09 06:32:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1960,103606,Ather Grid Point - Moto Store & Cafe,12.987554,77.620871,Bangalore,Karnataka,560042,(Unknown Operator),Operational,True,3.0,1,False,2018-05-04 08:14:00+00:00,2018-05-11 11:06:00+00:00
1961,103604,"International Tech Park, Bangalore",12.985421,77.738513,Bangalore,Karnataka,560066,(Unknown Operator),Operational,True,3.0,2,False,2018-05-04 08:06:00+00:00,2018-05-11 11:06:00+00:00
1962,103603,Ather Grid Point - Ascendas Park square mall,12.987219,77.736474,Bangalore,Karnataka,560066,(Unknown Operator),Operational,True,3.0,2,False,2018-05-04 08:01:00+00:00,2018-05-11 11:06:00+00:00
1963,96047,Tellus Power,17.413362,78.426597,Hyderabad,Telangana,500034,Unknown Operator,Operational,True,7.2,1,False,2017-12-07 10:12:00+00:00,2017-12-08 08:52:00+00:00


In [31]:
#Check unique appointment status values- unique()
df_chargers["town"].unique()

<ArrowStringArray>
[                     'MIRYALGUDA',                           'Ashta',
                         'Unknown',                        'Durgapur',
                         'Kevadia',                        'Palanpur',
                       'Ahmedabad',                      'Ahmedabad ',
                         'Bharuch',                          'Ambaji',
 ...
                        'Vandalur',                 'Irungattukottai',
            ' Chikkanayakanahalli',                'Mukundgarh Mandi',
                       'Vaigundam', 'Chennai City Corporation Limits',
                        'Guwahati',                       'Darbhanga',
                        'Clappana',                          'Nagpur']
Length: 755, dtype: str

In [32]:
# ============================================
# CELL: Light cleaning for 'town' — whitespace and casing only
# ============================================
df_chargers["town"] = df_chargers["town"].str.strip().str.title()

# Recheck how many unique values remain after just this simple fix
print("Unique towns after basic cleanup:", df_chargers["town"].nunique())

Unique towns after basic cleanup: 734


In [33]:
df_chargers["operator"].unique()

<ArrowStringArray>
[               '(Unknown Operator)',             'Charge_iN by Mahindra',
                    'ChargeMod (IN)',      '(Business Owner at Location)',
                        'Tata Power',                       'Statiq (IN)',
                     'Nikol EV (IN)',                     'Zeon Charging',
                'Chargezone (India)',               'Shell Recharge (IN)',
              'JIO BP Pulse (India)',                 'Adani Gas-EV (TR)',
                        'GO EC (IN)',                   'Bolt.Earth (IN)',
                  'Unknown Operator',                  'EarthtronEV (IN)',
                  'eDrive BPCL (IN)',                         'Tryk (IN)',
                       'Xobolt (IN)',                      'Volttic (IN)',
               'Hydra Charging (IN)',            'Relux Electric (India)',
                         'EESL (IN)',                           'Ecoplug',
 'GE WattStation (No longer active)',                   'Joulepoint (IN)',
      

In [34]:
# ============================================
# CELL: Merge duplicate "Unknown Operator" variants
# ============================================
df_chargers["operator"] = df_chargers["operator"].replace({
    "(Unknown Operator)": "Unknown Operator"
})

print("Unique operators after fix:", df_chargers["operator"].nunique())

Unique operators after fix: 34


In [35]:
df_chargers["status"].unique()

<ArrowStringArray>
[               'Operational',            'Not Operational',
    'Temporarily Unavailable', 'Partly Operational (Mixed)',
    'Planned For Future Date']
Length: 5, dtype: str

In [36]:
df_chargers["is_operational"].unique()

array([ True, False])

In [37]:
df_chargers["has_fast_charging"].unique()

array([ True, False])

In [38]:
print("Final cleaned shape:", df_chargers.shape)

Final cleaned shape: (1835, 15)


In [39]:
df_chargers= df_chargers.reset_index(drop = True)

In [40]:
df_chargers.to_csv("../outputs/chargers_cleaned.csv", index=False)
print("Saved successfully!")

Saved successfully!


In [41]:
df_chargers

,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
0,509422,Narketpally - Addanki - Medarametla Road,16.890620,79.561206,Miryalguda,Telangana,508207,Unknown Operator,Operational,True,180.0,6,True,2026-09-16 10:37:00+00:00,2026-09-19 10:39:00+00:00
1,504729,Uplug Ev Charging Station & Network,16.945913,74.405395,Ashta,Maharashtra,416301,Unknown Operator,Operational,True,60.0,2,True,2026-08-15 06:26:00+00:00,2026-08-18 06:28:00+00:00
2,503818,NH-7 Hotel,20.634579,78.927604,Unknown,Maharashtra,Unknown,Charge_iN by Mahindra,Operational,True,240.0,4,True,2026-08-09 13:23:00+00:00,2026-08-09 13:23:00+00:00
3,503599,The Big Bay Highway Pitstop,13.605916,77.786870,Unknown,Karnataka,562104,Charge_iN by Mahindra,Operational,True,240.0,4,True,2026-08-08 09:33:00+00:00,2026-08-08 09:33:00+00:00
4,502497,TransEV Charging Station,23.547824,87.272638,Durgapur,West Bengal,713203,Unknown Operator,Operational,True,60.0,4,True,2026-08-06 06:28:00+00:00,2026-08-09 06:32:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1830,103606,Ather Grid Point - Moto Store & Cafe,12.987554,77.620871,Bangalore,Karnataka,560042,Unknown Operator,Operational,True,3.0,1,False,2018-05-04 08:14:00+00:00,2018-05-11 11:06:00+00:00
1831,103604,"International Tech Park, Bangalore",12.985421,77.738513,Bangalore,Karnataka,560066,Unknown Operator,Operational,True,3.0,2,False,2018-05-04 08:06:00+00:00,2018-05-11 11:06:00+00:00
1832,103603,Ather Grid Point - Ascendas Park square mall,12.987219,77.736474,Bangalore,Karnataka,560066,Unknown Operator,Operational,True,3.0,2,False,2018-05-04 08:01:00+00:00,2018-05-11 11:06:00+00:00
1833,96047,Tellus Power,17.413362,78.426597,Hyderabad,Telangana,500034,Unknown Operator,Operational,True,7.2,1,False,2017-12-07 10:12:00+00:00,2017-12-08 08:52:00+00:00


In [42]:
df_chargers["state"].value_counts()

state
Kerala                    595
Karnataka                 378
Odisha                    284
Tamil Nadu                211
Maharashtra                85
Gujarat                    66
Rajasthan                  47
Delhi                      34
Haryana                    28
Uttar Pradesh              28
Telangana                  17
Uttarakhand                14
Madhya Pradesh             13
West Bengal                 9
Andhra Pradesh              7
Goa                         6
Dadra And Nagar Haveli      3
Assam                       2
Jharkhand                   2
Punjab                      2
Jammu And Kashmir           1
Himachal Pradesh            1
Puducherry                  1
Bihar                       1
Name: count, dtype: int64

In [43]:
southern_states = [
    "Kerala",
    "Karnataka",
    "Tamil Nadu",
    "Telangana",
    "Andhra Pradesh",
    "Puducherry"
]

In [45]:
df_south = df_chargers[df_chargers["state"].isin(southern_states)].copy()

In [47]:
print("Original dataset:", len(df_chargers))
print("Southern India dataset:", len(df_south))

print("\nSouthern India state-wise count:")
print(df_south["state"].value_counts())

Original dataset: 1835
Southern India dataset: 1209

Southern India state-wise count:
state
Kerala            595
Karnataka         378
Tamil Nadu        211
Telangana          17
Andhra Pradesh      7
Puducherry          1
Name: count, dtype: int64


In [48]:
df_south.to_csv(
    "../data/dataset_1_ev_charging_southern_india.csv",
    index=False
)